# 04 · Counterfactual Recommender — How to Fix Defects
Given a NOK part, this notebook answers: *"What setpoints should the operator change?"*

Two complementary strategies:
1. **Taguchi Nearest-OK** — find the experiment cell with lowest NOK rate closest in setpoint space
2. **Gradient Counterfactual** — gradient descent on SET_* features until model predicts OK

This is the *actionable intelligence* layer — unique research contribution.

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
import config as C
from data.loaders    import load_parquet, build_cv_index, join_cv_to_parquet, load_tabular_vector
from data.dataset    import MultiModalDataset, collate_fn, get_tabular_cols, get_dxp_cols
from models.fusion   import InjectionMouldingFusionModel
from training.splits import build_experiment_label_map, random_split
from analysis.counterfactual import (
    TaguchiBestCell, GradientCounterfactual, counterfactual_report,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
df     = load_parquet(C.PARQUET_PATH)
cv_idx = build_cv_index()
df     = join_cv_to_parquet(df, cv_idx)
tabular_cols  = get_tabular_cols(df)
dxp_cols      = get_dxp_cols(df)
exp_label_map = build_experiment_label_map(df)
train_df, _, test_df = random_split(df)

scaler   = StandardScaler()
tab_vals = [load_tabular_vector(r, tabular_cols) for _, r in train_df.iterrows()]
scaler.fit(np.array(tab_vals))
tab_med  = {c: float(train_df[c].median()) for c in tabular_cols if c in train_df.columns}
roi_med  = {c: float(train_df[c].median()) for c in C.THERMAL_ROI_COLS if c in train_df.columns}

model = InjectionMouldingFusionModel(
    tabular_in_dim=len(tabular_cols),
    n_dxp_channels=len(dxp_cols),
    n_experiments=len(exp_label_map),
).to(DEVICE)
ckpt = C.MODELS_DIR / "fusion_final.pt"
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
model.eval()

# Build Taguchi recommender on training data
recommender = TaguchiBestCell(train_df, nok_threshold=0.2)
print(f"Taguchi recommender: {len(recommender.ok_cells)} OK cells identified out of {train_df[C.GROUP_COL].nunique()} total")

## 1 · Select a NOK Part

In [ ]:
ds_test = MultiModalDataset(test_df, tabular_cols, dxp_cols, train=False,
                            tabular_medians=tab_med, roi_medians=roi_med, tabular_scaler=scaler)
ld_test = DataLoader(ds_test, batch_size=1, shuffle=False, collate_fn=collate_fn)

label_idx = C.LABEL_COLS.index(C.PRIMARY_TARGET)
nok_batch, nok_row_idx = None, None

for i, (batch, (_, row)) in enumerate(zip(ld_test, test_df.iterrows())):
    lbl = batch["labels"][0, label_idx].item()
    with torch.no_grad():
        b   = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = model(b)
        prob = torch.sigmoid(out["defect_logits"][0, label_idx]).item()
    if lbl == 1 and prob > 0.5:
        nok_batch   = batch
        nok_row     = row
        nok_prob    = prob
        nok_row_idx = i
        print(f"NOK part found: idx={i}  cycle={row.get(C.CYCLE_ID_COL,'?')}  P(NOK)={prob:.4f}")
        break

if nok_batch is None:
    print("No high-confidence NOK part found — using first test part for demonstration")
    for batch, (_, row) in zip(ld_test, test_df.iterrows()):
        nok_batch, nok_row = batch, row; break

## 2 · Taguchi Nearest-OK Recommendations

In [ ]:
recs = recommender.recommend(nok_row, top_k=3)

print("=" * 62)
print("TAGUCHI NEAREST-OK SETPOINT RECOMMENDATIONS")
print("=" * 62)
for rank, rec in enumerate(recs, 1):
    print(f"\n  Option {rank}  [cell={rec['experiment']}, NOK rate={rec['nok_rate']:.1%}, dist={rec['distance']:.3f}]")
    for col, delta in rec["deltas"].items():
        if abs(delta) < 0.01:
            continue
        sign = "+" if delta > 0 else ""
        print(f"    {col.replace('SET_',''):25s} {sign}{delta:+.2f}  → {rec['recommended'][col]:.2f}")

# Visualise top recommendation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
top_rec = recs[0]

for ax_i, rec in enumerate(recs):
    cols_with_delta = {k: v for k, v in rec["deltas"].items() if abs(v) > 0.01}
    if not cols_with_delta:
        ax = axes[ax_i]; ax.text(0.5, 0.5, "No significant change", ha="center", transform=ax.transAxes)
        continue
    short_names = [k.replace("SET_","") for k in cols_with_delta.keys()]
    deltas      = list(cols_with_delta.values())
    bar_cols    = ["#EF5350" if d < 0 else "#66BB6A" for d in deltas]
    axes[ax_i].barh(short_names, deltas, color=bar_cols, edgecolor="white")
    axes[ax_i].axvline(0, color="black", linewidth=0.8)
    axes[ax_i].set_title(f"Option {ax_i+1}: cell {rec['experiment']}\nNOK rate={rec['nok_rate']:.1%}")
    axes[ax_i].set_xlabel("Delta from current setpoint")

plt.suptitle("Taguchi Nearest-OK: Setpoint Changes Needed (green=increase, red=decrease)", fontsize=12)
plt.tight_layout(); plt.savefig(C.PLOTS_DIR/"counterfactual_taguchi.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 · Gradient Counterfactual Optimisation

In [ ]:
gcf = GradientCounterfactual(
    model=model, tabular_cols=tabular_cols,
    lambda_reg=0.3, lr=0.02, max_steps=300, target_prob=0.25,
)

cf_delta, info = gcf.run(nok_batch, DEVICE)

print("=" * 55)
print("GRADIENT COUNTERFACTUAL RESULT")
print("=" * 55)
print(f"  Status   : {'SUCCESS' if info['success'] else 'Did not fully converge'}")
print(f"  P(NOK)   : {info['nok_prob_original']:.4f}  →  {info['nok_prob_final']:.4f}")
print(f"  Steps    : {info['n_steps']}")
print()
print(gcf.describe(cf_delta, scaler.mean_, scaler.scale_))
print("=" * 55)

In [ ]:
# Visualise probability trajectory during optimisation (re-run with trace)
import torch.nn as nn

prob_trace = []
model.eval()
scaler_mean = scaler.mean_
scaler_std  = scaler.scale_
set_idx     = [i for i, c in enumerate(tabular_cols) if c.startswith("SET_")]
orig_tab    = nok_batch["tabular"].to(DEVICE).clone()

delta_free = torch.zeros((1, len(set_idx)), device=DEVICE, requires_grad=True)
opt        = torch.optim.Adam([delta_free], lr=0.02)

for step in range(300):
    opt.zero_grad()
    mod_tab = orig_tab.clone()
    for j, idx in enumerate(set_idx):
        mod_tab[:, idx] = orig_tab[:, idx] + delta_free[:, j]
    b2 = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in nok_batch.items()}
    b2["tabular"] = mod_tab
    out2  = model(b2)
    logit = out2["defect_logits"][0, label_idx]
    bce   = nn.functional.binary_cross_entropy_with_logits(logit.unsqueeze(0), torch.zeros(1,device=DEVICE))
    reg   = 0.3 * (delta_free**2).mean()
    (bce + reg).backward()
    opt.step()
    with torch.no_grad():
        prob_trace.append(torch.sigmoid(logit).item())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(prob_trace, color="#E53935", linewidth=2)
ax1.axhline(0.5, ls="--", color="gray", label="Decision boundary (0.5)")
ax1.axhline(0.25, ls="--", color="green", label="Target (0.25)")
ax1.fill_between(range(len(prob_trace)), prob_trace, 0.5,
                 where=[p < 0.5 for p in prob_trace], alpha=0.15, color="green", label="Below threshold")
ax1.set_xlabel("Optimisation step"); ax1.set_ylabel("P(NOK)")
ax1.set_title("Gradient CF: NOK Probability vs Step")
ax1.legend(); ax1.set_ylim(0, 1.05); ax1.grid(True, alpha=0.3)

# Final deltas (physical units)
final_delta_phys = delta_free[0].detach().cpu().numpy() * scaler_std[set_idx]
set_names = [tabular_cols[i].replace("SET_","") for i in set_idx]
bar_cols  = ["#66BB6A" if d > 0 else "#EF5350" for d in final_delta_phys]
ax2.barh(set_names, final_delta_phys, color=bar_cols, edgecolor="white")
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Setpoint delta (physical units)")
ax2.set_title("Gradient CF: Recommended Setpoint Changes")

plt.suptitle("Model-Guided Counterfactual: Minimum-Change Path to OK", fontsize=12)
plt.tight_layout(); plt.savefig(C.PLOTS_DIR/"counterfactual_gradient.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · Full Operator Report

In [ ]:
report = counterfactual_report(
    model=model, batch=nok_batch, query_row=nok_row,
    training_df=train_df, tabular_cols=tabular_cols,
    device=DEVICE, scaler_mean=scaler.mean_, scaler_std=scaler.scale_,
)
print(report)